# 05 — Frozen 100K C/P3/Q3 on full LCC

Configure paths, then run all cells. Uses existing 00d bboxes; no detector, training, contract or SHA preflight.

In [1]:
# Change only these paths and optional filters before running on Kaggle.
RESOURCE_ROOT='/kaggle/input/datasets/maithanhphuong/full-resources-v3/minifasnet_binary_crossdomain_resources_v2_lcc_pooled'
SCALE100K_ROOT='/kaggle/input/datasets/maithanhphuong/100k-output/binary_crossdomain_scale100k_C_P3_Q3_v1'
LCC_ROOT='/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD'
MINIFASNET_SOURCE_PATH='/kaggle/input/datasets/maithanhphuong/minifasnet/MiniFASNet.py'
OUTPUT_ROOT='/kaggle/working/full_lcc_100k_C_P3_Q3_evaluation_v1'
MODEL_FILTER=None  # None = C, P3, Q3; or a list of run IDs.
LCC_SPLIT_FILTER=None  # None = all; or a list from training, development, evaluation, combined.
REUSE_EXISTING=True
EVAL_BATCH_SIZE=256
NUM_WORKERS=4


In [2]:
import importlib.util, json, math, os, sys, zipfile
from pathlib import Path
import cv2, numpy as np, pandas as pd, torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, roc_curve
from torch.utils.data import Dataset, DataLoader
from IPython.display import FileLink, display

MODEL_IDS=('C100k_copy_pretrained','P3_100k_worst_ce25','Q3_100k_random_ce25_kl01')
PHYSICAL_SPLITS=('training','development','evaluation')
ALL_SPLITS=PHYSICAL_SPLITS+('combined',)
EXPECTED_COUNTS={'training':(8299,1223,7076),'development':(2948,405,2543),
                 'evaluation':(7580,314,7266),'combined':(18827,1942,16885)}
CROP_SCALE=2.7; INPUT_SIZE=80

def select(value,allowed,name):
    if value is None:return tuple(allowed)
    if not isinstance(value,(list,tuple)) or not value or len(set(value))!=len(value) or set(value)-set(allowed):
        raise ValueError(name+' must contain unique known values')
    return tuple(value)
models=select(MODEL_FILTER,MODEL_IDS,'MODEL_FILTER')
splits=select(LCC_SPLIT_FILTER,ALL_SPLITS,'LCC_SPLIT_FILTER')
needed=PHYSICAL_SPLITS if 'combined' in splits else tuple(s for s in PHYSICAL_SPLITS if s in splits)
if EVAL_BATCH_SIZE<1 or NUM_WORKERS<0:raise ValueError('Invalid batch or worker count')
resource=Path(RESOURCE_ROOT);scale_root=Path(SCALE100K_ROOT);lcc_root=Path(LCC_ROOT)
source=Path(MINIFASNET_SOURCE_PATH);out=Path(OUTPUT_ROOT)
for path in (resource,scale_root,lcc_root):
    if not path.is_dir():raise FileNotFoundError(path)
if not source.is_file():raise FileNotFoundError(source)
out.mkdir(parents=True,exist_ok=True)
def read_json(path):return json.loads(Path(path).read_text())
def write_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(value,indent=2,sort_keys=True,allow_nan=False)+'\n')
def write_csv(path,frame):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    frame.to_csv(path,index=False)
def file_stamp(path):
    stat=Path(path).stat()
    return {'path':str(Path(path).resolve()),'size':stat.st_size,'mtime_ns':stat.st_mtime_ns}

def require_record(rec,key):
    if rec.get('status')!='VALID' or rec.get('usable_for_pad') is not True or rec.get('bbox_origin')!='SCRFD' or rec.get('source_key')!=key or not str(rec.get('bbox_source','')).startswith('SCRFD'):raise RuntimeError('Unscorable/non-SCRFD: '+key)
    box=rec.get('bbox_xyxy')
    if box is None or len(box)!=4 or 'crop_factor' in rec:raise RuntimeError('Not a raw bbox: '+key)
    x1,y1,x2,y2=map(float,box)
    if not np.isfinite(box).all() or x2<=x1 or y2<=y1:raise RuntimeError('Invalid bbox: '+key)
    return rec

def mini_crop_bgr(image,bbox_xyxy,scale=CROP_SCALE,size=INPUT_SIZE):
    # Exact 03b CropImage-compatible geometry.
    src_h,src_w=image.shape[:2];x1,y1,x2,y2=map(float,bbox_xyxy);box_w,box_h=x2-x1,y2-y1
    if box_w<=0 or box_h<=0 or src_w<2 or src_h<2:raise ValueError('Invalid crop geometry')
    used=min((src_h-1)/box_h,(src_w-1)/box_w,float(scale));new_w,new_h=box_w*used,box_h*used
    cx,cy=box_w/2+x1,box_h/2+y1;left,top=cx-new_w/2,cy-new_h/2;right,bottom=cx+new_w/2,cy+new_h/2
    if left<0:right-=left;left=0
    if top<0:bottom-=top;top=0
    if right>src_w-1:left-=right-src_w+1;right=src_w-1
    if bottom>src_h-1:top-=bottom-src_h+1;bottom=src_h-1
    lx,ty,rx,by=map(int,(left,top,right,bottom))
    if lx<0 or ty<0 or rx>=src_w or by>=src_h or rx<lx or by<ty:raise RuntimeError('Crop escaped image')
    patch=image[ty:by+1,lx:rx+1]
    if patch.size==0:raise RuntimeError('Empty crop')
    return cv2.resize(patch,(size,size))

def tensor_of(patch):
    if patch.shape!=(80,80,3) or patch.dtype!=np.uint8:raise ValueError('Expected BGR uint8 80x80')
    return torch.from_numpy(np.ascontiguousarray(patch.transpose(2,0,1))).float().div_(255.)

class PADNet(nn.Module):
    def __init__(self,copied):
        super().__init__();self.copied=copied
        self.trunk=module.MiniFASNetV2(embedding_size=128,conv6_kernel=(5,5),drop_p=.2,num_classes=3,img_channel=3)
        if copied:
            self.trunk.prob=nn.Identity()
            self.pad_head=nn.Linear(128,2);self.spoof_type_head=nn.Linear(128,11)
            self.lighting_head=nn.Linear(128,5);self.attributes_head=nn.Linear(128,40)
    def features(self,x):
        t=self.trunk
        for layer in (t.conv1,t.conv2_dw,t.conv_23,t.conv_3,t.conv_34,t.conv_4,t.conv_45,t.conv_5,t.conv_6_sep,t.conv_6_dw,t.conv_6_flatten):x=layer(x)
        if t.embedding_size!=512:x=t.linear(x)
        x=t.drop(t.bn(x))
        if x.ndim!=2 or x.shape[1]!=128:raise RuntimeError('MiniFAS representation is not [B,128]')
        return x
    def forward(self,x,aux=False):
        z=self.features(x)
        if not self.copied:return self.trunk.prob(z)
        pad=self.pad_head(z)
        if aux:return pad,self.spoof_type_head(z),self.lighting_head(z),self.attributes_head(z)
        return pad

def pad_score(logits):
    z=torch.as_tensor(logits)
    if z.shape[1]==3:return (z[:,0]-torch.logsumexp(z[:,1:],dim=1)).float().cpu().numpy()
    if z.shape[1]==2:return (z[:,0]-z[:,1]).float().cpu().numpy()
    raise ValueError('Expected two or three PAD logits')

def metrics(y3,scores,threshold,candidate_n):
    y=np.asarray(y3,dtype=int);s=np.asarray(scores,dtype=float);real=y==0
    if len(y)!=len(s) or set(real)!={False,True} or not np.isfinite(s).all():raise ValueError('Invalid metric population')
    fpr,tpr,thr=roc_curve(real.astype(int),s);eer_idx=int(np.argmin(abs(fpr-(1-tpr))))
    pred=s>=threshold;apcer=float(pred[~real].mean());bpcer=float((~pred[real]).mean())
    return {'auc':float(roc_auc_score(real.astype(int),s)),'eer':float(.5*(fpr[eer_idx]+1-tpr[eer_idx])),
      'tpr_at_fpr_1pct':float(np.interp(.01,fpr,tpr)),'apcer':apcer,'bpcer':bpcer,'acer':.5*(apcer+bpcer),
      'hter':.5*(apcer+bpcer),'binary_accuracy':float((pred==real).mean()),'locked_source_threshold':float(threshold),
      'candidate_n':int(candidate_n),'scored_n':len(y),'detector_coverage':len(y)/candidate_n}

manifest_names={s:('lcc_official_evaluation_full' if s=='evaluation' else 'lcc_'+s+'_full') for s in ALL_SPLITS}
cache_names={s:('lcc_official_evaluation_bbox_cache' if s=='evaluation' else 'lcc_'+s+'_bbox_cache') for s in ALL_SPLITS}
manifest_paths={s:resource/'manifests'/(manifest_names[s]+'.csv') for s in ALL_SPLITS}
cache_paths={s:resource/'bbox'/(cache_names[s]+'.json') for s in ALL_SPLITS}
frames={};records={};coverage={}
for split in ALL_SPLITS:
    manifest=manifest_paths[split];cache_path=cache_paths[split]
    if not manifest.is_file() or not cache_path.is_file():raise FileNotFoundError((manifest,cache_path))
    frame=pd.read_csv(manifest,keep_default_na=False)
    required={'path','label','status','usable_for_pad','bbox_origin'}
    if not required<=set(frame.columns) or (split=='combined' and 'lcc_split' not in frame):
        raise RuntimeError('Unexpected manifest columns: '+split)
    n,nreal,nspoof=EXPECTED_COUNTS[split]
    if len(frame)!=n or int((frame.label==0).sum())!=nreal or int((frame.label==1).sum())!=nspoof or frame.path.duplicated().any():
        raise RuntimeError('Unexpected LCC candidates: '+split)
    cache=read_json(cache_path)['records']
    if set(cache)!=set(frame.path):raise RuntimeError('Manifest/cache paths differ: '+split)
    usable=frame.usable_for_pad.astype(bool)
    for row in frame.itertuples(index=False):
        rec=cache[row.path]
        if (row.status,bool(row.usable_for_pad),row.bbox_origin)!=(rec.get('status'),rec.get('usable_for_pad'),rec.get('bbox_origin')):
            raise RuntimeError('Manifest/cache row differs: '+row.path)
        if bool(row.usable_for_pad):require_record(rec,row.path)
    frames[split]=frame;records[split]=cache
    coverage[split]={'candidate_n':n,'scored_n':int(usable.sum()),'coverage':float(usable.mean()),
      'real_candidate_n':nreal,'real_scored_n':int((usable&(frame.label==0)).sum()),
      'spoof_candidate_n':nspoof,'spoof_scored_n':int((usable&(frame.label==1)).sum())}
combined=pd.concat([frames[s].assign(lcc_split=s) for s in PHYSICAL_SPLITS],ignore_index=True)
if not frames['combined'].equals(combined):raise RuntimeError('Combined manifest differs from physical splits')
print('Frozen LCC candidates / usable rows:',{s:(c['candidate_n'],c['scored_n']) for s,c in coverage.items()})


Frozen LCC candidates / usable rows: {'training': (8299, 8283), 'development': (2948, 2947), 'evaluation': (7580, 7564), 'combined': (18827, 18794)}


In [3]:
spec=importlib.util.spec_from_file_location('local_minifasnet_v2',source)
module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp=device.type=='cuda'
run_info={}
for run_id in models:
    folder=scale_root/'runs'/run_id
    best_path=folder/'best.pth';freeze_path=folder/'evaluation_freeze.json'
    if not best_path.is_file() or not freeze_path.is_file():raise FileNotFoundError((best_path,freeze_path))
    freeze=read_json(freeze_path)
    threshold=float(freeze['locked_threshold']);epoch=int(freeze['selected_epoch'])
    if freeze.get('run_id')!=run_id or not math.isfinite(threshold):raise RuntimeError('Invalid frozen threshold: '+run_id)
    best=torch.load(best_path,map_location='cpu',weights_only=True)
    if best.get('run_id')!=run_id or best.get('epoch')!=epoch or not isinstance(best.get('state_dict'),dict):
        raise RuntimeError('Checkpoint does not match frozen run/epoch: '+run_id)
    run_info[run_id]={'folder':folder,'best_path':best_path,'state':best['state_dict'],
                      'threshold':threshold,'epoch':epoch,'freeze_path':freeze_path}
    print(run_id,'epoch',epoch,'Val15K locked threshold',threshold)
print('Device:',device,'batch:',EVAL_BATCH_SIZE,'AMP:',use_amp)

class LCCData(Dataset):
    def __init__(self,split):
        self.split=split
        self.rows=frames[split].loc[frames[split].usable_for_pad.astype(bool)].reset_index(drop=True)
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i];key=str(row.path)
        path=(lcc_root/'LCC_FASD_evaluation'/key) if self.split=='evaluation' else lcc_root/key
        image=cv2.imread(str(path),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(path)
        patch=mini_crop_bgr(image,require_record(records[self.split][key],key)['bbox_xyxy'])
        return tensor_of(patch),int(row.label),key

def load_model(run_id):
    model=PADNet(True)
    model.load_state_dict(run_info[run_id]['state'],strict=True)
    return model.to(device).eval()

def infer_split(model,split):
    dataset=LCCData(split)
    loader=DataLoader(dataset,batch_size=EVAL_BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,
                      pin_memory=use_amp,drop_last=False)
    keys=[];logits=[]
    with torch.no_grad():
        for x,_,paths in loader:
            with torch.cuda.amp.autocast(enabled=use_amp):z=model(x.to(device,non_blocking=True))
            keys.extend(paths);logits.append(z.float().cpu().numpy())
    expected=dataset.rows.path.astype(str).tolist()
    if keys!=expected:raise RuntimeError('Inference order changed: '+split)
    if not logits:raise RuntimeError('No usable images: '+split)
    z=np.concatenate(logits)
    if z.shape!=(len(expected),2) or not np.isfinite(z).all():raise RuntimeError('Invalid logits: '+split)
    return z

def predictions(split,z,threshold):
    frame=frames[split].copy().assign(lcc_split=split)
    usable=frame.usable_for_pad.astype(bool).to_numpy()
    scores=pad_score(z)
    if len(scores)!=int(usable.sum()):raise RuntimeError('Score count differs: '+split)
    frame['score']=np.nan;frame['prediction_at_locked_threshold']=np.nan
    frame['logit_0']=np.nan;frame['logit_1']=np.nan
    frame.loc[usable,'score']=scores
    frame.loc[usable,'prediction_at_locked_threshold']=(scores>=threshold).astype(int)
    frame.loc[usable,'logit_0']=z[:,0];frame.loc[usable,'logit_1']=z[:,1]
    frame['locked_threshold']=threshold
    return frame

def validate_prediction(pred,split,threshold):
    base=frames[split]
    required={'path','lcc_split','label','status','usable_for_pad','bbox_origin','score',
              'prediction_at_locked_threshold','locked_threshold'}
    if not required<=set(pred.columns) or len(pred)!=len(base):raise RuntimeError('Prediction shape differs: '+split)
    for col in ('path','label','status','usable_for_pad','bbox_origin'):
        if pred[col].tolist()!=base[col].tolist():raise RuntimeError('Prediction rows differ: '+split+' '+col)
    expected=base.lcc_split.tolist() if split=='combined' else [split]*len(base)
    if pred.lcc_split.tolist()!=expected:raise RuntimeError('Prediction split differs: '+split)
    usable=base.usable_for_pad.astype(bool).to_numpy()
    scores=pd.to_numeric(pred.score,errors='coerce').to_numpy()
    labels=pd.to_numeric(pred.prediction_at_locked_threshold,errors='coerce').to_numpy()
    th=pd.to_numeric(pred.locked_threshold,errors='coerce').to_numpy()
    if (not np.isfinite(scores[usable]).all() or not np.isnan(scores[~usable]).all() or
        not np.isnan(labels[~usable]).all() or not np.array_equal(labels[usable],(scores[usable]>=threshold).astype(float)) or
        not np.all(th==threshold)):
        raise RuntimeError('Prediction scores/threshold differ: '+split)
    return scores,usable

def score_metrics(pred,split,threshold):
    scores,usable=validate_prediction(pred,split,threshold)
    result=metrics(pred.loc[usable,'label'],scores[usable],threshold,len(pred))
    result.update({'split':split,**{k:v for k,v in coverage[split].items() if k not in ('candidate_n','scored_n','coverage')}})
    return result

def official_check(run_id,result):
    path=run_info[run_id]['folder']/'lcc_metrics.json'
    if not path.is_file():return 'original metric file unavailable'
    old=read_json(path)
    for key in ('candidate_n','scored_n','locked_source_threshold'):
        if result[key]!=old[key]:raise RuntimeError('Original evaluation count/threshold differs: '+run_id+' '+key)
    keys=('auc','eer','tpr_at_fpr_1pct','apcer','bpcer','acer','hter','binary_accuracy')
    delta=max(abs(result[k]-old[k]) for k in keys)
    if not math.isfinite(delta) or delta>1e-6:raise RuntimeError('Original evaluation metrics differ: '+run_id+' '+str(delta))
    return 'matched (max difference '+str(delta)+')'


C100k_copy_pretrained epoch 7 Val15K locked threshold 0.0103759765625
P3_100k_worst_ce25 epoch 11 Val15K locked threshold -0.1998291015625
Q3_100k_random_ce25_kl01 epoch 10 Val15K locked threshold 0.649139404296875
Device: cuda batch: 256 AMP: True


In [4]:
results={};checks={}
for run_id in models:
    info=run_info[run_id];folder=out/run_id
    folder.mkdir(parents=True,exist_ok=True)
    model=None;physical={};results[run_id]={}
    for split in needed:
        path=folder/f'lcc_{split}_predictions.csv'
        sidecar=folder/f'lcc_{split}_inputs.json'
        stamp={'checkpoint':file_stamp(info['best_path']),'freeze':file_stamp(info['freeze_path']),
               'manifest':file_stamp(manifest_paths[split]),'cache':file_stamp(cache_paths[split]),
               'source':file_stamp(source),'lcc_root':str(lcc_root.resolve()),
               'threshold':info['threshold'],'batch_size':EVAL_BATCH_SIZE,'amp':use_amp}
        pred=None
        if REUSE_EXISTING and path.is_file() and sidecar.is_file() and read_json(sidecar)==stamp:
            try:
                candidate=pd.read_csv(path)
                validate_prediction(candidate,split,info['threshold'])
                pred=candidate;print(run_id,split,'reused')
            except (ValueError,KeyError,RuntimeError):pass
        if pred is None:
            if model is None:model=load_model(run_id)
            pred=predictions(split,infer_split(model,split),info['threshold'])
            validate_prediction(pred,split,info['threshold'])
            write_csv(path,pred);write_json(sidecar,stamp)
            print(run_id,split,'inferred',coverage[split]['scored_n'],'images')
        physical[split]=pred
        result=score_metrics(pred,split,info['threshold'])
        results[run_id][split]=result
        write_json(folder/f'lcc_{split}_metrics.json',result)
        if split=='evaluation':checks[run_id]=official_check(run_id,result)
    if 'combined' in splits:
        pooled=pd.concat([physical[s] for s in PHYSICAL_SPLITS],ignore_index=True)
        result=score_metrics(pooled,'combined',info['threshold'])
        write_csv(folder/'lcc_combined_predictions.csv',pooled)
        write_json(folder/'lcc_combined_metrics.json',result)
        results[run_id]['combined']=result
    write_json(folder/'evaluation_provenance.json',{'run_id':run_id,'best_epoch':info['epoch'],
        'locked_val15k_threshold':info['threshold'],'checkpoint':str(info['best_path']),
        'evaluated_splits':list(results[run_id]),'original_evaluation_check':checks.get(run_id,'not requested')})
    if model is not None:del model
    if use_amp:torch.cuda.empty_cache()

rows=[{'model':run_id,'split':split,**metric} for run_id in models for split,metric in results[run_id].items()]
write_csv(out/'splitwise_summary.csv',pd.DataFrame(rows));write_json(out/'splitwise_summary.json',rows)
comparison=[]
for run_id in models:
    e=results[run_id].get('evaluation');c=results[run_id].get('combined')
    row={'model':run_id}
    for key in ('auc','eer','hter','tpr_at_fpr_1pct'):
        row['evaluation_'+key]=e[key] if e else None
        row['combined_'+key]=c[key] if c else None
        row['combined_minus_evaluation_'+key]=c[key]-e[key] if e and c else None
    comparison.append(row)
write_csv(out/'comparison_summary.csv',pd.DataFrame(comparison));write_json(out/'comparison_summary.json',comparison)

pair_defs=(('P3-C',MODEL_IDS[1],MODEL_IDS[0]),('Q3-C',MODEL_IDS[2],MODEL_IDS[0]),('Q3-P3',MODEL_IDS[2],MODEL_IDS[1]))
deltas=[]
for split in ALL_SPLITS:
    for name,left,right in pair_defs:
        if left in results and right in results and split in results[left] and split in results[right]:
            deltas.append({'split':split,'comparison':name,**{k+'_delta':results[left][split][k]-results[right][split][k]
                           for k in ('auc','eer','hter','tpr_at_fpr_1pct')}})
write_csv(out/'cross_model_deltas.csv',pd.DataFrame(deltas,columns=['split','comparison','auc_delta','eer_delta','hter_delta','tpr_at_fpr_1pct_delta']))
write_json(out/'cross_model_deltas.json',deltas)

report=['# Frozen C/P3/Q3 on full LCC','',
        'Evaluation only. Each model uses its own locked CelebA-Val15K threshold. No target selection or calibration.','',
        '| Model | Split | Candidates | Scored | AUC | EER | HTER | TPR@FPR1% |',
        '|---|---|---:|---:|---:|---:|---:|---:|']
for row in rows:
    report.append(f"| {row['model']} | {row['split']} | {row['candidate_n']} | {row['scored_n']} | {row['auc']:.6f} | {row['eer']:.6f} | {row['hter']:.6f} | {row['tpr_at_fpr_1pct']:.6f} |")
report+=['','## Evaluation versus pooled LCC','']
for row in comparison:
    delta=row['combined_minus_evaluation_auc']
    report.append(f"- {row['model']}: AUC change {delta:+.6f}." if delta is not None else f"- {row['model']}: both splits required.")
report+=['','## Pairwise model changes','']
for row in deltas:report.append(f"- {row['split']} {row['comparison']}: AUC {row['auc_delta']:+.6f}; EER {row['eer_delta']:+.6f}; HTER {row['hter_delta']:+.6f}.")
report+=['','## Scientific questions','','Q1. LCC training versus official evaluation:']
for run_id in models:
    a=results[run_id].get('training');b=results[run_id].get('evaluation')
    report.append(f"- {run_id}: AUC {a['auc']-b['auc']:+.6f}; HTER {a['hter']-b['hter']:+.6f}." if a and b else f'- {run_id}: requires both splits.')
report+=['','Q2. LCC development versus official evaluation:']
for run_id in models:
    a=results[run_id].get('development');b=results[run_id].get('evaluation')
    report.append(f"- {run_id}: AUC {a['auc']-b['auc']:+.6f}; HTER {a['hter']-b['hter']:+.6f}." if a and b else f'- {run_id}: requires both splits.')
report+=['','Q3. Pooled versus official evaluation:']
for row in comparison:
    a=row['combined_minus_evaluation_auc'];h=row['combined_minus_evaluation_hter']
    report.append(f"- {row['model']}: AUC {a:+.6f}; HTER {h:+.6f}." if a is not None else f"- {row['model']}: requires both splits.")
report+=['','Q4. C/P3/Q3 ordering by AUC:']
for split in ALL_SPLITS:
    report.append('- '+split+': '+', '.join(sorted(MODEL_IDS,key=lambda m:results[m][split]['auc'],reverse=True)) if all(m in results and split in results[m] for m in MODEL_IDS) else '- '+split+': requires all three models.')
report+=['','Q5. Target population composition and legacy comparison:','',
 'The pooled-versus-evaluation differences above measure the effect of target population for these frozen models. They can partly explain a legacy gap if material, but do not reproduce legacy AUC because detector, crop, image size, normalization, backbone, and training recipe also differ.',
 'No target result selects a new checkpoint, model, or threshold.']
(out/'full_lcc_report.md').write_text('\n'.join(report)+'\n')
write_json(out/'evaluation_config.json',{'model_filter':list(models),'split_filter':list(splits),
    'batch_size':EVAL_BATCH_SIZE,'num_workers':NUM_WORKERS,'reuse_existing':REUSE_EXISTING,
    'preprocessing':'2.7x raw bbox crop; 80x80 BGR float /255; score z_real-z_attack'})
write_json(out/'provenance.json',{'resource_root':str(resource),'scale100k_root':str(scale_root),
    'lcc_root':str(lcc_root),'model_source':str(source),'torch':torch.__version__,
    'cuda':torch.version.cuda,'device':str(device),'models':list(models),'splits':list(splits)})
(out/'FULL_LCC_100K_EVALUATION_PROTOCOL_v1.md').write_text("# Full LCC 100K evaluation protocol v1\n\nEvaluate the frozen `C100k_copy_pretrained`, `P3_100k_worst_ce25`, and `Q3_100k_random_ce25_kl01` checkpoints on LCC training, development, official evaluation, and their concatenation. Load each run's `best.pth` and its own `evaluation_freeze.json` Val15K threshold. Do not train, choose checkpoints, or fit thresholds on LCC.\n\nRead the completed 00d manifests and SCRFD bbox caches. Score only `VALID`, `usable_for_pad=True` rows with a raw SCRFD bbox; retain failed detections as unscored candidates. Use notebook 04's MiniFASNetV2 wrapper, 2.7× crop, 80×80 BGR float input divided by 255, `z_real-z_attack` score, and metric definitions. No detector is run.\n\nTraining/development paths are relative to the common LCC root; official evaluation paths are relative to its `LCC_FASD_evaluation` directory. Infer each physical split once per model. Build combined results by concatenating physical predictions and recomputing metrics from pooled scores. Report candidate/scored counts, AUC, EER, TPR@FPR1%, APCER, BPCER, HTER, and accuracy. If an original 100K official-evaluation metric file is available, require agreement within 1e-6.\n\n`MODEL_FILTER` and `LCC_SPLIT_FILTER` can restrict work. A combined request uses all three physical splits. `REUSE_EXISTING` reuses a prediction only when its input file stamps, threshold, inference settings, and candidate rows match; otherwise that split is recomputed. No contract or SHA audit is required. The pooled diagnostic does not select a model or reproduce legacy model/preprocessing results.\n")
zip_path=out.parent/(out.name+'.zip')
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
    for path in sorted(out.rglob('*')):
        if path.is_file() and (path.parent==out or path.relative_to(out).parts[0] in models):
            archive.write(path,(Path(out.name)/path.relative_to(out)).as_posix())
print(pd.DataFrame(rows)[['model','split','candidate_n','scored_n','auc','eer','hter']].to_string(index=False))
display(FileLink(str(zip_path)))


/tmp/ipykernel_23/4129941707.py:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):z=model(x.to(device,non_blocking=True))


C100k_copy_pretrained training inferred 8283 images


/tmp/ipykernel_23/4129941707.py:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):z=model(x.to(device,non_blocking=True))


C100k_copy_pretrained development inferred 2947 images


/tmp/ipykernel_23/4129941707.py:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):z=model(x.to(device,non_blocking=True))


...

/tmp/ipykernel_23/4129941707.py:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):z=model(x.to(device,non_blocking=True))


Q3_100k_random_ce25_kl01 development inferred 2947 images


/tmp/ipykernel_23/4129941707.py:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):z=model(x.to(device,non_blocking=True))


Q3_100k_random_ce25_kl01 evaluation inferred 7564 images
                   model       split  candidate_n  scored_n      auc      eer     hter
   C100k_copy_pretrained    training         8299      8283 0.839086 0.251233 0.254693
   C100k_copy_pretrained development         2948      2947 0.809007 0.276549 0.323655
   C100k_copy_pretrained  evaluation         7580      7564 0.730801 0.321380 0.344573
   C100k_copy_pretrained    combined        18827     18794 0.800096 0.282207 0.294675
      P3_100k_worst_ce25    training         8299      8283 0.856367 0.231783 0.237068
      P3_100k_worst_ce25 development         2948      2947 0.789960 0.298281 0.321839
      P3_100k_worst_ce25  evaluation         7580      7564 0.737351 0.331053 0.343067
      P3_100k_worst_ce25    combined        18827     18794 0.811892 0.269913 0.280320
Q3_100k_random_ce25_kl01    training         8299      8283 0.844808 0.240674 0.245612
Q3_100k_random_ce25_kl01 development         2948      2947 0.759032 0.31

/kaggle/working/full_lcc_100k_C_P3_Q3_evaluation_v1.zip